# NB-02: ペース係数・クラス別par_time推定

**目的**: NB-01の出力（adjusted_time_sec）をベースに以下の2つを推定する：
1. **Δpace係数（coeff_pace）**: ペース偏差から走破タイムへの係数（venue×surface×distance単位）
2. **par_time_class**: クラス別2着馬基準タイム（venue×surface×distance×class_rank単位）

## 入力
| ファイル | 説明 |
|---|---|
| `output/nb01/megu_dataset.parquet` | NB-01出力・全クリーニング済みデータ |
| `output/nb01/par_splits.parquet` | 基準前半スプリット（distance×surface×track_cat） |

## 出力
| ファイル | 説明 |
|---|---|
| `output/nb02/coeff_pace.parquet` | venue×surface×distance単位ペース係数 |
| `output/nb02/par_time_class.parquet` | venue×surface×distance×class_rank単位par_time |

## 1. セットアップ

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge

# 日本語フォント設定（NB-01と同様）
plt.rcParams['axes.unicode_minus'] = False
_JP_FONT_CANDIDATES = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc',
]
_jp_font_path = next((p for p in _JP_FONT_CANDIDATES if Path(p).exists()), None)
if _jp_font_path:
    fm.fontManager.addfont(_jp_font_path)
    _jp_font_name = fm.FontProperties(fname=_jp_font_path).get_name()
    plt.rcParams['font.family'] = _jp_font_name
    plt.rcParams['font.sans-serif'] = [_jp_font_name, 'DejaVu Sans']
    print(f'日本語フォント: {_jp_font_name}')
else:
    print('WARNING: 日本語フォント未検出。`sudo apt-get install -y fonts-noto-cjk` を実行してください。')

INPUT_DIR  = Path('/home/jovyan/work/keiba-vpn/notebooks/megu_index/output/nb01')
OUTPUT_DIR = Path('/home/jovyan/work/keiba-vpn/notebooks/megu_index/output/nb02')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup OK')

## 2. データロード

In [ ]:
# ── megu_dataset / par_splits ロード ─────────────────────────────────────
print('megu_dataset.parquet をロード中...')
df = pd.read_parquet(INPUT_DIR / 'megu_dataset.parquet')
print(f'  megu_dataset: {df.shape}  期間: {df["date"].min()} 〜 {df["date"].max()}')

print('par_splits.parquet をロード中...')
par_splits = pd.read_parquet(INPUT_DIR / 'par_splits.parquet')
print(f'  par_splits:   {par_splits.shape}')
print(f'  par_splits 列: {par_splits.columns.tolist()}')

# ── distance の型を int に統一 ────────────────────────────────────────────
df['distance'] = pd.to_numeric(df['distance'], errors='coerce').astype('Int64')
par_splits['distance'] = pd.to_numeric(par_splits['distance'], errors='coerce').astype('Int64')

# ── par_splits を distance×surface×track_cat でマージして front_split_dev を計算 ──
# par_splits の列名確認: distance, surface, track_cat, par_front_split_final, n
df = df.merge(
    par_splits[['distance', 'surface', 'track_cat', 'par_front_split_final']],
    on=['distance', 'surface', 'track_cat'],
    how='left',
)
# front_split_dev = front_split_sec - par_front_split_final（欠損は0埋め）
df['front_split_dev'] = (
    df['front_split_sec'] - df['par_front_split_final']
).fillna(0.0)

print(f'\nfront_split_dev 統計:')
print(df['front_split_dev'].describe())

# ── 学習データ: 2020〜2024 ────────────────────────────────────────────────
df_train = df[df['year'] <= 2024].copy()
print(f'\n学習データ（year<=2024）: {len(df_train):,} 行')

## 3. Δpace係数の推定

### 3-1. セル単位OLS（venue×surface×distance）

In [ ]:
# ── レース内偏差（within-race demean）────────────────────────────────────
# fitデータ: front_split_sec が notna かつ distance_band が notna
df_pace_fit = df_train[
    df_train['front_split_sec'].notna() &
    df_train['distance_band'].notna() &
    df_train['adjusted_time_sec'].notna()
].copy()

# within-race demean
df_pace_fit['front_split_dev_dm'] = (
    df_pace_fit['front_split_dev']
    - df_pace_fit.groupby('race_id')['front_split_dev'].transform('mean')
)
df_pace_fit['time_dm'] = (
    df_pace_fit['adjusted_time_sec']
    - df_pace_fit.groupby('race_id')['adjusted_time_sec'].transform('mean')
)

print(f'pace fit データ: {len(df_pace_fit):,} 行')

# ── セル単位OLS ──────────────────────────────────────────────────────────
# OLS: slope = Cov(front_split_dev_dm, time_dm) / Var(front_split_dev_dm)
pace_cell_rows = []
for (venue, surface, distance), grp in df_pace_fit.groupby(
    ['venue', 'surface', 'distance'], sort=False
):
    x = grp['front_split_dev_dm'].to_numpy()
    y = grp['time_dm'].to_numpy()
    n = len(x)
    varx = np.var(x)
    if n >= 30 and varx > 1e-9:
        cov = np.cov(x, y, bias=True)[0, 1]
        slope = cov / varx
    else:
        slope = np.nan
    pace_cell_rows.append({
        'venue': venue,
        'surface': surface,
        'distance': int(distance),
        'coeff_pace_cell': slope,
        'n_fit': n,
    })

df_coeff_cell = pd.DataFrame(pace_cell_rows)
# n>=30 かつ slope>0 のみ採用
df_coeff_cell['valid_cell'] = (
    (df_coeff_cell['n_fit'] >= 30) &
    (df_coeff_cell['coeff_pace_cell'] > 0)
)
print(f'セル数 (venue×surface×distance): {len(df_coeff_cell):,}')
print(f'  有効セル (n>=30 and slope>0): {df_coeff_cell["valid_cell"].sum():,}')
print(f'  無効セル (フォールバック必要): {(~df_coeff_cell["valid_cell"]).sum():,}')

### 3-2. フォールバック（n<30 または slope<=0）

In [ ]:
# ── distance_band を df_pace_fit に付与 ──────────────────────────────────
# megu_dataset の distance_band は df_train に既存
df_pace_fit_band = df_pace_fit.merge(
    df_train[['race_id', 'horse_id', 'distance_band']].drop_duplicates(),
    on=['race_id', 'horse_id'],
    how='left',
    suffixes=('', '_y'),
)
if 'distance_band' not in df_pace_fit.columns:
    df_pace_fit['distance_band'] = df_pace_fit_band['distance_band']

# ── フォールバック1: surface × distance_band ─────────────────────────────
pool_distband_rows = []
for (surface, distband), grp in df_pace_fit.groupby(
    ['surface', 'distance_band'], sort=False
):
    x = grp['front_split_dev_dm'].to_numpy()
    y = grp['time_dm'].to_numpy()
    n = len(x)
    varx = np.var(x)
    if n >= 30 and varx > 1e-9:
        cov = np.cov(x, y, bias=True)[0, 1]
        slope = cov / varx
    else:
        slope = np.nan
    pool_distband_rows.append({
        'surface': surface,
        'distance_band': distband,
        'coeff_pace_distband': slope if (not pd.isna(slope) and slope > 0) else np.nan,
    })

df_pool_distband = pd.DataFrame(pool_distband_rows)
print('フォールバック1 (surface×distance_band):')
print(df_pool_distband.to_string())

# ── フォールバック2: surface 全体 ─────────────────────────────────────────
pool_surface_rows = []
for surface, grp in df_pace_fit.groupby('surface', sort=False):
    x = grp['front_split_dev_dm'].to_numpy()
    y = grp['time_dm'].to_numpy()
    n = len(x)
    varx = np.var(x)
    if n >= 30 and varx > 1e-9:
        cov = np.cov(x, y, bias=True)[0, 1]
        slope = cov / varx
    else:
        slope = np.nan
    pool_surface_rows.append({
        'surface': surface,
        'coeff_pace_surface': slope if (not pd.isna(slope) and slope > 0) else np.nan,
    })

df_pool_surface = pd.DataFrame(pool_surface_rows)
print('\nフォールバック2 (surface全体):')
print(df_pool_surface.to_string())

### 3-3. 係数の統合・制約

In [ ]:
# ── distance_band を df_coeff_cell に付与（surface×distanceからマッピング） ──
dist_to_band = (
    df_pace_fit[['distance', 'distance_band']]
    .drop_duplicates()
    .dropna(subset=['distance_band'])
)
df_coeff_cell = df_coeff_cell.merge(
    dist_to_band,
    on='distance',
    how='left',
)

# ── フォールバック適用 ────────────────────────────────────────────────────
df_coeff_cell = df_coeff_cell.merge(
    df_pool_distband,
    on=['surface', 'distance_band'],
    how='left',
)
df_coeff_cell = df_coeff_cell.merge(
    df_pool_surface,
    on='surface',
    how='left',
)

# 優先順位: cell > pool_distband > pool_surface
def _select_coeff(row):
    if row['valid_cell'] and not pd.isna(row['coeff_pace_cell']):
        return float(row['coeff_pace_cell']), 'cell'
    elif not pd.isna(row.get('coeff_pace_distband')):
        return float(row['coeff_pace_distband']), 'pool_distband'
    elif not pd.isna(row.get('coeff_pace_surface')):
        return float(row['coeff_pace_surface']), 'pool_surface'
    else:
        return np.nan, 'none'

results = df_coeff_cell.apply(_select_coeff, axis=1)
df_coeff_cell['coeff_pace_raw'] = [r[0] for r in results]
df_coeff_cell['source'] = [r[1] for r in results]

# ── 係数の制約: 0.3〜1.5 にクリップ ─────────────────────────────────────
df_coeff_cell['coeff_pace'] = df_coeff_cell['coeff_pace_raw'].clip(lower=0.3, upper=1.5)

# ── 出力DataFrame: coeff_pace ──────────────────────────────────────────
coeff_pace = df_coeff_cell[['venue', 'surface', 'distance', 'coeff_pace', 'n_fit', 'source']].copy()
coeff_pace['distance'] = coeff_pace['distance'].astype(int)
coeff_pace['n_fit'] = coeff_pace['n_fit'].astype(int)

print('coeff_pace サマリー:')
print(coeff_pace.shape)
print(coeff_pace['source'].value_counts())
print(coeff_pace['coeff_pace'].describe())

# ── ヒストグラムで分布確認 ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
coeff_pace['coeff_pace'].hist(bins=30, ax=ax, color='steelblue', alpha=0.7, edgecolor='white')
ax.set_title('coeff_pace 分布（0.3〜1.5クリップ後）')
ax.set_xlabel('coeff_pace')
ax.set_ylabel('セル数')
ax.axvline(0.3, color='red', linestyle='--', linewidth=1, label='下限 0.3')
ax.axvline(1.5, color='orange', linestyle='--', linewidth=1, label='上限 1.5')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coeff_pace_dist.png', dpi=80)
plt.show()
print('coeff_pace ヒストグラム保存完了')

## 4. クラスランクの付与

### 4-1. クラスランク定義 / 4-2. grade列からのランク抽出

In [ ]:
# ── クラスランク定義 ──────────────────────────────────────────────────────
CLASS_RANK = {
    '未勝利': 1, '新馬': 1,
    '1勝': 2,
    '2勝': 3,
    '3勝': 4,
    'OP': 5, 'L': 5,
    'G3': 6, 'G2': 6,
    'G1': 7,
}

def assign_class_rank(grade, race_class):
    """grade列を基本に CLASS_RANK でマッピング。
    マッチしない場合はrace_classの文字列パターンでフォールバック。
    """
    grade_str = str(grade) if pd.notna(grade) else ''
    # gradeが直接マッチ
    if grade_str in CLASS_RANK:
        return CLASS_RANK[grade_str]
    # gradeがG1/G2/G3パターン
    for key in ['G1', 'G2', 'G3', 'L', 'OP', '3勝', '2勝', '1勝', '未勝利', '新馬']:
        if key in grade_str:
            return CLASS_RANK[key]
    # race_classでフォールバック
    rc = str(race_class) if pd.notna(race_class) else ''
    if '新馬' in rc:
        return CLASS_RANK['新馬']
    if '未勝利' in rc:
        return CLASS_RANK['未勝利']
    if '1勝' in rc:
        return CLASS_RANK['1勝']
    if '2勝' in rc:
        return CLASS_RANK['2勝']
    if '3勝' in rc:
        return CLASS_RANK['3勝']
    if 'G1' in rc:
        return CLASS_RANK['G1']
    if 'G2' in rc:
        return CLASS_RANK['G2']
    if 'G3' in rc:
        return CLASS_RANK['G3']
    if 'L' in rc or 'OP' in rc or 'オープン' in rc:
        return CLASS_RANK['OP']
    return np.nan

df['class_rank'] = df.apply(
    lambda r: assign_class_rank(r['grade'], r['race_class']), axis=1
)

print('class_rank 分布:')
print(df['class_rank'].value_counts(dropna=False).sort_index().to_dict())
print(f'class_rank カバレッジ: {df["class_rank"].notna().mean():.1%}')

### 4-3. 世代戦の除外フィルタ

In [ ]:
# ── 世代戦除外フィルタ（3歳以上 or 4歳以上の混合条件戦のみ） ───────────────────
# 2歳のみ・3歳のみを除外
exclude_mask = (
    df['race_class'].str.contains('２歳|2歳', na=False) |
    (
        df['race_class'].str.contains('３歳|3歳', na=False) &
        ~df['race_class'].str.contains('以上', na=False)
    )
)
df_open = df[~exclude_mask].copy()

print(f'全データ:       {len(df):,} 行')
print(f'世代戦除外:     {exclude_mask.sum():,} 行 ({exclude_mask.mean():.1%})')
print(f'df_open（混合）: {len(df_open):,} 行')
print(f'\ndf_open class_rank 分布:')
print(df_open['class_rank'].value_counts(dropna=False).sort_index().to_dict())

## 5. par_time線形回帰

### 5-1. 使用データの準備

In [ ]:
# ── 2着馬のみ、class_rank 1〜7、year<=2024 ──────────────────────────────
df_par_base = df_open[
    (df_open['finish_pos'] == 2) &
    (df_open['class_rank'].between(1, 7)) &
    (df_open['year'] <= 2024) &
    df_open['adjusted_time_sec'].notna()
].copy()

print(f'par_time 学習データ（2着馬、year<=2024）: {len(df_par_base):,} 行')
print(f'class_rank 分布:')
print(df_par_base['class_rank'].value_counts().sort_index().to_dict())
print(f'\nユニーク venue×surface×distance セル数: '
      f'{df_par_base.groupby(["venue","surface","distance"]).ngroups}')

### 5-2. セル単位線形回帰（venue×surface×distance）

In [ ]:
# ── グローバルβ（Ridge）の事前推定 ────────────────────────────────────────
# グローバルβ: 全データを venue×surface×distance でセンタリングしてから
# class_rank → adjusted_time_sec の Ridge 回帰
df_par_base['time_mean_cell'] = df_par_base.groupby(
    ['venue', 'surface', 'distance']
)['adjusted_time_sec'].transform('mean')
df_par_base['time_center'] = df_par_base['adjusted_time_sec'] - df_par_base['time_mean_cell']

X_global = df_par_base[['class_rank']].values
y_global = df_par_base['time_center'].values
ridge_global = Ridge(alpha=1.0, fit_intercept=True)
ridge_global.fit(X_global, y_global)
global_beta = float(ridge_global.coef_[0])
print(f'グローバルβ（Ridge, alpha=1.0）: {global_beta:.4f}')
print(f'  （betaは負であるべき: G1が最速）: {"OK" if global_beta < 0 else "WARNING: 正値"}')

# ── 距離帯ごとのプーリングβ（フォールバック1用） ─────────────────────────
pool_distband_beta = {}
for (surface, distband), grp in df_par_base.groupby(['surface', 'distance_band'], sort=False):
    if grp['class_rank'].nunique() >= 2 and len(grp) >= 10:
        # セル平均をキャンセルして beta のみ
        grp_c = grp.copy()
        grp_c['time_cell'] = grp_c.groupby(['venue', 'distance'])['adjusted_time_sec'].transform('mean')
        grp_c['time_c2'] = grp_c['adjusted_time_sec'] - grp_c['time_cell']
        X_db = grp_c[['class_rank']].values
        y_db = grp_c['time_c2'].values
        ridge_db = Ridge(alpha=1.0, fit_intercept=False)
        ridge_db.fit(X_db, y_db)
        pool_distband_beta[(surface, str(distband))] = float(ridge_db.coef_[0])
    else:
        pool_distband_beta[(surface, str(distband))] = global_beta

print('\nプーリングβ（surface×distance_band）:')
for k, v in sorted(pool_distband_beta.items()):
    print(f'  {k}: {v:.4f}')

In [ ]:
# ── セル単位線形回帰 ──────────────────────────────────────────────────────
# 各セルで class_rank別 adjusted_time_sec の平均を計算し、
# class_rank水準が2以上あるセルのみ LinearRegression でfit

par_time_rows = []

for (venue, surface, distance), grp in df_par_base.groupby(
    ['venue', 'surface', 'distance'], sort=False
):
    cell_mean = grp.groupby('class_rank')['adjusted_time_sec'].mean()
    n_levels = len(cell_mean)
    n_fit = len(grp)
    
    distance_int = int(distance)
    distband_key = str(grp['distance_band'].iloc[0]) if 'distance_band' in grp.columns else None
    
    if n_levels >= 2:
        # セル単位回帰
        X_cell = np.array(cell_mean.index).reshape(-1, 1).astype(float)
        y_cell = cell_mean.values.astype(float)
        lr = LinearRegression(fit_intercept=True)
        lr.fit(X_cell, y_cell)
        alpha_val = float(lr.intercept_)
        beta_val  = float(lr.coef_[0])
        source = 'cell'
    else:
        # フォールバック: beta はプーリング or グローバル
        if distband_key and (surface, distband_key) in pool_distband_beta:
            beta_val = pool_distband_beta[(surface, distband_key)]
            source = 'pool_distband'
        else:
            beta_val = global_beta
            source = 'pool_surface'
        # alpha はセル単位の mean から推定
        cell_time_mean = grp['adjusted_time_sec'].mean()
        cell_rank_mean = grp['class_rank'].mean()
        alpha_val = cell_time_mean - beta_val * cell_rank_mean
    
    # par_time_sec を class_rank=1〜7 について生成
    for cr in range(1, 8):
        par_time_rows.append({
            'venue': venue,
            'surface': surface,
            'distance': distance_int,
            'class_rank': cr,
            'par_time_sec': alpha_val + beta_val * cr,
            'alpha': alpha_val,
            'beta': beta_val,
            'n_fit': n_fit,
            'source': source,
        })

par_time_class = pd.DataFrame(par_time_rows)
par_time_class['distance'] = par_time_class['distance'].astype(int)
par_time_class['class_rank'] = par_time_class['class_rank'].astype(int)
par_time_class['n_fit'] = par_time_class['n_fit'].astype(int)

print(f'par_time_class shape: {par_time_class.shape}')
print(f'source 分布:')
print(par_time_class['source'].value_counts())
print(f'\npar_time_class サンプル（芝 東京 2000m）:')
sample = par_time_class[
    (par_time_class['surface'] == '芝') &
    (par_time_class['venue'].str.contains('東京', na=False)) &
    (par_time_class['distance'] == 2000)
]
if len(sample):
    print(sample[['venue','surface','distance','class_rank','par_time_sec','alpha','beta','n_fit','source']].to_string(index=False))
else:
    print('（東京芝2000mのデータなし）')

### 5-4. 品質確認

In [ ]:
# ── beta>0 のセルをフラグ ─────────────────────────────────────────────────
beta_pos_mask = par_time_class['beta'] > 0
bad_cells = par_time_class[beta_pos_mask][['venue','surface','distance','beta','source']].drop_duplicates()
print(f'beta>0（異常）のセル数: {len(bad_cells):,}')
if len(bad_cells) > 0:
    print('異常セル（先頭10件）:')
    print(bad_cells.head(10).to_string(index=False))

# ── 1勝クラス（rank=2）と G1（rank=7）の par_time 差の分布 ─────────────────
pt_rank2 = par_time_class[par_time_class['class_rank'] == 2][['venue','surface','distance','par_time_sec']].rename(columns={'par_time_sec': 'pt_rank2'})
pt_rank7 = par_time_class[par_time_class['class_rank'] == 7][['venue','surface','distance','par_time_sec']].rename(columns={'par_time_sec': 'pt_rank7'})
pt_diff = pt_rank2.merge(pt_rank7, on=['venue','surface','distance'])
pt_diff['diff_2vs7'] = pt_diff['pt_rank2'] - pt_diff['pt_rank7']

print(f'\n1勝クラス(rank=2) vs G1(rank=7) par_time差 分布:')
print(pt_diff['diff_2vs7'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pt_diff['diff_2vs7'].hist(bins=30, ax=axes[0], color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('par_time差：1勝クラス - G1 (秒)')
axes[0].set_xlabel('秒')
axes[0].set_ylabel('セル数')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)

# source 別カバレッジ
src_counts = par_time_class[['venue','surface','distance','source']].drop_duplicates()['source'].value_counts()
src_counts.plot.bar(ax=axes[1], color=['#4878CF','#6ACC65','#D65F5F'], alpha=0.8)
axes[1].set_title('source 別カバレッジ（venue×surface×distance セル）')
axes[1].set_xlabel('source')
axes[1].set_ylabel('セル数')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'par_time_quality.png', dpi=80)
plt.show()
print('品質確認プロット保存完了')

## 6. 保存

In [ ]:
# ── coeff_pace.parquet ────────────────────────────────────────────────────
# スキーマ: venue, surface, distance, coeff_pace, n_fit, source
coeff_pace_out = coeff_pace[['venue', 'surface', 'distance', 'coeff_pace', 'n_fit', 'source']].copy()
path_coeff = OUTPUT_DIR / 'coeff_pace.parquet'
coeff_pace_out.to_parquet(path_coeff, index=False)
print(f'coeff_pace 保存完了: {path_coeff}')
print(f'  shape: {coeff_pace_out.shape}')
print(coeff_pace_out.head())

# ── par_time_class.parquet ────────────────────────────────────────────────
# スキーマ: venue, surface, distance, class_rank, par_time_sec, alpha, beta, n_fit, source
par_time_out = par_time_class[[
    'venue', 'surface', 'distance', 'class_rank',
    'par_time_sec', 'alpha', 'beta', 'n_fit', 'source'
]].copy()
path_par = OUTPUT_DIR / 'par_time_class.parquet'
par_time_out.to_parquet(path_par, index=False)
print(f'\npar_time_class 保存完了: {path_par}')
print(f'  shape: {par_time_out.shape}')
print(par_time_out.head())

print('\n=== NB-02 完了 ===')
print(f'  coeff_pace:     {coeff_pace_out.shape}')
print(f'  par_time_class: {par_time_out.shape}')